In [1]:
# Install required libraries for the Gradio app, model inference, Hugging Face access, and PDF generation
!pip -q install -U gradio transformers accelerate sentencepiece safetensors huggingface_hub reportlab

# Import core libraries for file handling, inference, temporary files, timestamps, PDF export, and Gradio UI
import os
import csv
import gc
import torch
import tempfile
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

# Import Gradio, Transformers model utilities, and Hugging Face login helpers
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from google.colab import userdata

# Load the Hugging Face token from Colab secrets and authenticate the session
HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

# Define the fine-tuned KG-to-text model and the base general chat model IDs
KG_MODEL_ID = "BSVGK/gemma-1.1-2b-it-drugbank-kg2text-merged-v2"
CHAT_MODEL_ID = "google/gemma-1.1-2b-it"

# Detect the available device and choose the best supported precision for inference
device = "cuda" if torch.cuda.is_available() else "cpu"
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16_supported else (torch.float16 if device == "cuda" else torch.float32)

# Print the selected device and datatype configuration
print(f"Using device: {device}")
print(f"Using dtype: {dtype}")

# Print the KG model ID before loading the fine-tuned KG-to-text model
print(f"Loading KG model: {KG_MODEL_ID}")

# Load the tokenizer for the fine-tuned KG-to-text model
kg_tokenizer = AutoTokenizer.from_pretrained(
    KG_MODEL_ID,
    token=HF_TOKEN
)

# Set the KG tokenizer pad token to EOS if it is not already defined
if kg_tokenizer.pad_token is None:
    kg_tokenizer.pad_token = kg_tokenizer.eos_token

# Load the fine-tuned KG-to-text model with automatic device placement
kg_model = AutoModelForCausalLM.from_pretrained(
    KG_MODEL_ID,
    token=HF_TOKEN,
    dtype=dtype,
    device_map="auto"
)

# Set the KG model to evaluation mode for inference
kg_model.eval()
print("KG model loaded successfully.")

# Print the chat model ID before loading the general Gemma chat model
print(f"Loading Chat model: {CHAT_MODEL_ID}")

# Load the tokenizer for the general chat model
chat_tokenizer = AutoTokenizer.from_pretrained(
    CHAT_MODEL_ID,
    token=HF_TOKEN
)

# Set the chat tokenizer pad token to EOS if it is not already defined
if chat_tokenizer.pad_token is None:
    chat_tokenizer.pad_token = chat_tokenizer.eos_token

# Load the general chat model with automatic device placement
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_ID,
    token=HF_TOKEN,
    dtype=dtype,
    device_map="auto"
)

# Set the chat model to evaluation mode for inference
chat_model.eval()
print("Chat model loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.5 MB/s eta 0:00:00
Using device: cuda
Using dtype: torch.bfloat16
Loading KG model: BSVGK/gemma-1.1-2b-it-drugbank-kg2text-merged-v2


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/488 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.3M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Chat model loaded successfully.


In [2]:
# Define prompt templates, generation functions, file helpers, download utilities, chat logic, UI styling, and launch the Gradio interface
# Define the Gemma chat template tokens and generation limits used across KG-to-text and general chat inference
START_USER = "<start_of_turn>user\n"
START_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>"

KG_MAX_NEW_TOKENS = 400
KG_MIN_NEW_TOKENS = 100
CHAT_MAX_NEW_TOKENS = 220
MODEL_MAX_INPUT_LENGTH = 2048

# Build the fixed KG-to-text prompt that grounds generation only on the provided triples
def build_generation_prompt(triples_text: str) -> str:
    return (
        f"{START_USER}"
        f"You are DrugKG Text AI, a grounded biomedical knowledge graph to text generation system.\n\n"
        f"Task:\n"
        f"Convert the following DrugBank-style knowledge graph triples into a clear, complete, and accurate natural language description.\n\n"
        f"Rules:\n"
        f"- Use only the information present in the triples.\n"
        f"- Do not invent any facts.\n"
        f"- Keep the output fluent, grounded, and domain-focused.\n"
        f"- Produce the best complete answer directly.\n\n"
        f"Triples:\n"
        f"{triples_text.strip()}"
        f"{END_TURN}\n"
        f"{START_MODEL}"
    )

# Build the multi-turn prompt for the general chat assistant using prior conversation history
def build_general_chat_prompt(history, user_message: str) -> str:
    prompt = (
        f"{START_USER}"
        f"You are a helpful, clear, and professional assistant.\n"
        f"Answer the user's question directly and naturally."
        f"{END_TURN}\n"
        f"{START_MODEL}"
        f"Understood."
        f"{END_TURN}\n"
    )

    # Append previous user and assistant turns to preserve chat context
    for msg in history:
        if msg["role"] == "user":
            prompt += f"{START_USER}{msg['content']}{END_TURN}\n"
        elif msg["role"] == "assistant":
            prompt += f"{START_MODEL}{msg['content']}{END_TURN}\n"

    # Add the current user message and open the assistant turn for generation
    prompt += f"{START_USER}{user_message.strip()}{END_TURN}\n"
    prompt += f"{START_MODEL}"
    return prompt

# Generate text from a prepared prompt using the selected model and tokenizer
def generate_with_model(prompt: str, model, tokenizer, max_new_tokens: int, min_new_tokens: int = 1) -> str:
    # Tokenize the prompt with truncation to the maximum input length
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MODEL_MAX_INPUT_LENGTH
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Get the end-of-turn token ID to stop generation at the correct boundary
    end_turn_ids = tokenizer.encode(END_TURN, add_special_tokens=False)
    end_turn_id = end_turn_ids[0] if len(end_turn_ids) > 0 else tokenizer.eos_token_id

    # Run deterministic generation without sampling
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=end_turn_id
        )

    # Decode only the newly generated tokens after the prompt
    output_ids = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(output_ids, skip_special_tokens=False).strip()

    # Trim the output at the first end-of-turn token if present
    if END_TURN in generated_text:
        generated_text = generated_text.split(END_TURN, 1)[0].strip()

    # Clear memory after generation to reduce GPU pressure
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return generated_text.strip()

# Generate grounded KG-to-text output using the fine-tuned DrugKG model
def generate_kg_text(triples_text: str) -> str:
    prompt = build_generation_prompt(triples_text)
    return generate_with_model(
        prompt=prompt,
        model=kg_model,
        tokenizer=kg_tokenizer,
        max_new_tokens=KG_MAX_NEW_TOKENS,
        min_new_tokens=KG_MIN_NEW_TOKENS
    )

# Generate a general chat response using the base Gemma chat model
def generate_general_chat(history, user_message: str) -> str:
    prompt = build_general_chat_prompt(history, user_message)
    return generate_with_model(
        prompt=prompt,
        model=chat_model,
        tokenizer=chat_tokenizer,
        max_new_tokens=CHAT_MAX_NEW_TOKENS,
        min_new_tokens=1
    )

# Read uploaded triple files and return their text content
def read_uploaded_file(file_obj):
    if file_obj is None:
        return ""

    try:
        file_path = file_obj.name if hasattr(file_obj, "name") else file_obj
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except Exception as e:
        return f"Could not read uploaded file: {str(e)}"

# Load uploaded triple file content into the textbox and optionally append it to existing text
def load_triples_from_file(file_obj, existing_text):
    uploaded_text = read_uploaded_file(file_obj)

    if not uploaded_text:
        return existing_text or ""

    if existing_text and existing_text.strip():
        return f"{existing_text.strip()}\n\n{uploaded_text}"
    return uploaded_text

# Create a downloadable PDF file containing the input triples and generated output
def create_pdf_file(triples_text, generated_text):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    safe_time = datetime.now().strftime("%Y%m%d_%H%M%S")

    temp_dir = tempfile.mkdtemp()
    pdf_path = os.path.join(temp_dir, f"drugkg_output_{safe_time}.pdf")

    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4
    x = 40
    y = height - 40
    line_height = 14

    # Write titled text blocks into the PDF with automatic line wrapping and page breaks
    def write_block(title, text, y_pos):
        c.setFont("Helvetica-Bold", 12)
        c.drawString(x, y_pos, title)
        y_pos -= 18
        c.setFont("Helvetica", 10)

        for raw_line in str(text).split("\n"):
            line = raw_line if raw_line.strip() else " "
            while len(line) > 110:
                c.drawString(x, y_pos, line[:110])
                line = line[110:]
                y_pos -= line_height
                if y_pos < 60:
                    c.showPage()
                    y_pos = height - 40
                    c.setFont("Helvetica", 10)
            c.drawString(x, y_pos, line)
            y_pos -= line_height
            if y_pos < 60:
                c.showPage()
                y_pos = height - 40
                c.setFont("Helvetica", 10)
        y_pos -= 10
        return y_pos

    # Add the PDF title and generation timestamp
    c.setFont("Helvetica-Bold", 14)
    c.drawString(x, y, "DrugKG Text AI Output")
    y -= 25
    c.setFont("Helvetica", 10)
    c.drawString(x, y, f"Generated at: {timestamp}")
    y -= 25

    # Write the input triples and generated output blocks to the PDF
    y = write_block("INPUT TRIPLES", triples_text, y)
    y = write_block("GENERATED TEXT", generated_text, y)

    c.save()
    return pdf_path

# Create a downloadable CSV file containing the timestamp, triples, and generated text
def create_csv_file(triples_text, generated_text):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    safe_time = datetime.now().strftime("%Y%m%d_%H%M%S")

    temp_dir = tempfile.mkdtemp()
    csv_path = os.path.join(temp_dir, f"drugkg_output_{safe_time}.csv")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["generated_at", "triples", "generated_text"])
        writer.writerow([timestamp, triples_text, generated_text])

    return csv_path

# Combine textbox and uploaded file triples, generate output text, and prepare a downloadable file
def generate_and_prepare_download(triples_text, triples_file, download_format):
    file_text = read_uploaded_file(triples_file)
    combined_triples = ""

    # Merge manual textbox triples with uploaded file triples if both are provided
    if triples_text and triples_text.strip():
        combined_triples += triples_text.strip()

    if file_text and file_text.strip() and not file_text.startswith("Could not read uploaded file:"):
        if combined_triples:
            combined_triples += "\n\n"
        combined_triples += file_text.strip()

    # Return a message if no triples are available for generation
    if not combined_triples.strip():
        return "", None, "Please provide triples by text input or file upload."

    # Generate the grounded KG-to-text output
    generated_text = generate_kg_text(combined_triples)

    if not generated_text:
        generated_text = "No text was generated. Try shorter or clearer triples."

    # Create the selected download file format from the generated output
    if download_format == "PDF":
        output_file = create_pdf_file(combined_triples, generated_text)
    else:
        output_file = create_csv_file(combined_triples, generated_text)

    status = f"Generation completed successfully. Download format: {download_format}"
    return generated_text, output_file, status

# Generate a reply for the general chatbot tab and update the conversation history
def chatbot_reply(message, history):
    message = (message or "").strip()
    history = history or []

    if not message:
        return history, ""

    # Generate a chat response from the base Gemma model
    reply = generate_general_chat(history, message)

    if not reply:
        reply = "I could not generate a response."

    # Append the new user and assistant messages to the chat history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})

    return history, ""

# Reset all KG-to-text interface fields to their default state
def clear_all():
    return "", None, "PDF", "", None, "Cleared."

# Reset the chatbot conversation and clear the message box
def clear_chat():
    return [], ""

# Define the custom CSS styling used to control the visual theme of the Gradio interface
custom_css = """
body {
    background: linear-gradient(135deg, #0b1120, #111827, #1e293b);
    color: #e5e7eb;
}

.gradio-container {
    max-width: 1320px !important;
    margin: auto;
    background: transparent !important;
}

#hero {
    text-align: center;
    padding: 24px;
    border-radius: 22px;
    background: linear-gradient(135deg, rgba(15,23,42,0.96), rgba(30,41,59,0.96));
    border: 1px solid rgba(148, 163, 184, 0.18);
    box-shadow: 0 12px 36px rgba(0,0,0,0.35);
    margin-bottom: 20px;
}

.hero-title {
    font-size: 2.2rem;
    font-weight: 800;
    color: #f8fafc;
    margin-bottom: 8px;
    letter-spacing: 0.3px;
}

.hero-subtitle {
    font-size: 1.02rem;
    color: #e2e8f0;
    font-weight: 600;
    line-height: 1.6;
}

.card {
    border-radius: 20px !important;
    background: rgba(15, 23, 42, 0.95) !important;
    border: 1px solid rgba(148, 163, 184, 0.16) !important;
    box-shadow: 0 10px 28px rgba(0,0,0,0.28);
    padding: 14px !important;
}

.section-title {
    font-size: 1.15rem;
    font-weight: 700;
    color: #f8fafc;
    margin-bottom: 10px;
}

.footer-note {
    text-align: center;
    color: #94a3b8;
    font-size: 13px;
    margin-top: 12px;
}

textarea, input {
    background: #020617 !important;
    color: #e5e7eb !important;
    border-radius: 14px !important;
    border: 1px solid rgba(148, 163, 184, 0.18) !important;
}

button {
    border-radius: 14px !important;
}

#download_selector label {
    background: #0f172a !important;
    color: #cbd5e1 !important;
    border: 1px solid rgba(148, 163, 184, 0.22) !important;
    border-radius: 12px !important;
    padding: 10px 12px !important;
    margin-bottom: 8px !important;
}

#download_selector label:has(input:checked) {
    background: linear-gradient(135deg, #0ea5e9, #2563eb) !important;
    color: white !important;
    border: 1px solid #7dd3fc !important;
    box-shadow: 0 8px 20px rgba(37,99,235,0.28) !important;
}

#overview_block, #overview_block * {
    color: #e5e7eb !important;
    opacity: 1 !important;
    visibility: visible !important;
}

#overview_block ul {
    padding-left: 20px !important;
}

#overview_block li {
    margin-bottom: 10px !important;
}
"""

# Build the complete Gradio interface with tabs for KG-to-text generation, general chat, and system overview
with gr.Blocks(title="DrugKG Text AI") as demo:
    # Create the hero header section shown at the top of the interface
    with gr.Column(elem_id="hero"):
        gr.HTML("""
        <div class="hero-title">DrugKG Text AI</div>
        <div class="hero-subtitle">
            Advanced knowledge graph-to-text generation and general AI interaction interface
            built for accurate, structured, and high-clarity output from DrugBank-style triples.
        </div>
        """)

    # Create the main tabbed interface for the application
    with gr.Tabs():
        # Create the KG-to-text generation tab
        with gr.Tab("KG to Text Generation"):
            with gr.Row():
                # Create the left panel for triple input, file upload, and action controls
                with gr.Column(scale=1, elem_classes="card"):
                    gr.HTML('<div class="section-title">Input Knowledge Graph Triples</div>')

                    triples_text = gr.Textbox(
                        lines=16,
                        label="Paste Triples",
                        placeholder="Paste DrugBank-style triples here..."
                    )

                    triples_file = gr.File(
                        label="Upload Triple File",
                        file_types=[".txt", ".csv", ".json", ".jsonl", ".ttl", ".rdf", ".md"]
                    )

                    load_file_btn = gr.Button("Load Uploaded File Into Textbox")

                    download_format = gr.Radio(
                        choices=["PDF", "CSV"],
                        value="PDF",
                        label="Download Format",
                        elem_id="download_selector"
                    )

                    with gr.Row():
                        generate_btn = gr.Button("Generate Output", variant="primary")
                        clear_btn = gr.Button("Clear")

                # Create the right panel for generated output, downloadable file, and status
                with gr.Column(scale=1, elem_classes="card"):
                    gr.HTML('<div class="section-title">Generated Text</div>')
                    generated_output = gr.Textbox(
                        lines=16,
                        label="Output"
                    )

                    download_file = gr.File(label="Download Result File")

                    status_box = gr.Textbox(
                        label="Status",
                        interactive=False
                    )

            # Load uploaded file content into the triples textbox
            load_file_btn.click(
                fn=load_triples_from_file,
                inputs=[triples_file, triples_text],
                outputs=triples_text
            )

            # Generate text and prepare the requested download file
            generate_btn.click(
                fn=generate_and_prepare_download,
                inputs=[triples_text, triples_file, download_format],
                outputs=[generated_output, download_file, status_box]
            )

            # Clear all KG-to-text interface fields
            clear_btn.click(
                fn=clear_all,
                inputs=[],
                outputs=[triples_text, triples_file, download_format, generated_output, download_file, status_box]
            )

        # Create the general chat assistant tab
        with gr.Tab("General Chat Assistant"):
            with gr.Column(elem_classes="card"):
                gr.HTML('<div class="section-title">General AI Chat</div>')
                gr.Markdown("Ask general questions here. This tab uses the base Gemma instruction model.")
                chatbot = gr.Chatbot(height=440)
                msg = gr.Textbox(
                    placeholder="Ask a general question..."
                )
                with gr.Row():
                    send_btn = gr.Button("Send", variant="primary")
                    clear_chat_btn = gr.Button("Clear Conversation")

            # Send button triggers general chat response generation
            send_btn.click(
                fn=chatbot_reply,
                inputs=[msg, chatbot],
                outputs=[chatbot, msg]
            )

            # Enter key submission triggers general chat response generation
            msg.submit(
                fn=chatbot_reply,
                inputs=[msg, chatbot],
                outputs=[chatbot, msg]
            )

            # Clear the current chat history and message box
            clear_chat_btn.click(
                fn=clear_chat,
                inputs=[],
                outputs=[chatbot, msg]
            )

        # Create the system overview tab showing supported capabilities
        with gr.Tab("System Overview"):
            with gr.Column(elem_classes="card", elem_id="overview_block"):
                gr.HTML("""
                <div class="section-title">Platform Capabilities</div>
                <div>
                    <p>This interface supports:</p>
                    <ul>
                        <li>direct text input for drug knowledge graph triples</li>
                        <li>triple file upload as input</li>
                        <li>downloadable output file in PDF or CSV format</li>
                        <li>general chat-style interaction using a base instruction model</li>
                        <li>fixed best-answer KG-to-text generation using the merged fine-tuned model</li>
                    </ul>
                    <p><strong>Model architecture:</strong></p>
                    <ul>
                        <li><strong>KG to Text tab:</strong> uses your merged fine-tuned DrugBank model</li>
                        <li><strong>General Chat tab:</strong> uses the base Gemma instruction model</li>
                    </ul>
                </div>
                """)

    # Add the footer note below the interface
    gr.HTML('<div class="footer-note">Structured generation • general chat • complete fixed-output mode</div>')

# Launch the Gradio app with the custom theme and CSS styling
demo.launch(share=True, theme=gr.themes.Base(), css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a8c86f3ce31e149f5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
